# Advanced Auto-Encoders

## What are Autoencoders?

Autoencoders are the data encoding techniques based on Unsupervised Artificial Neural Networks. This special type of ANN is trained to encode the data so that in such a way that data is represented in compressed form. The Autoencoders are also trained to decode the data so that, the original data can be reconstructed as far as possible.

## Architecture of Autoencoders

 The architecture for autoencoders are varied. In this section LSTM autoencoders is discussed. LSTM based autoencoders are used to encode and decode the sequence data.

Why sequence data is challenging to process?

- Sequence data are challenging for prediction task because the size of the is not fixed but it varies.
- Also, the temporal series of the data representation make it challenging to extract the features.

So, the building a predictive model to predict the sequence data involve sequence of operation and hence such problems are called as Sequence-to Sequence. Autoencoders comes as the best choice to handle sequence-to-sequence problems.

## Outlier/Anomaly detection using Autoencoders:

Suppose the input data is highly correlated and requires a technique to detect the anomaly or an outlier then, Autoencoders is the best choice. Since, autoencoders can encode the data in the compressed format, they can handle the correlated data.

Let’s train the autoencoders using MNIST data set using simple Feed Forward neural network.

## Simple 6 layered Feed Forward Autoencoders built to train on MNIST data

Once the autoencoders is trained on MNIST data set, an anomaly detection can be done using 2 different images. First one of the images from the MNIST data set is chosen and feed to the trained autoencoders. Since, this image is not an anomaly, the error or loss function is expected to be very low. Next, when some random image is given as test image, the loss rate is expected to be very high as it is an anomaly.

### Installing Packages

In [ ]:
$pip install numpy tensorflow keras matplotlib

### Importing Libraries  

In [ ]:
import numpy as np
import keras
from keras.datasets import mnist
from keras.models import Sequential, Model
from keras.layers import Dense, Input
from keras import optimizers
from keras.optimizers import Adam
from keras.preprocessing import image

### Building Model

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
train_x = x_train.reshape(60000, 784) / 255
val_x = x_test.reshape(10000, 784) / 255
autoencoder = Sequential()
autoencoder.add(Dense(512,  activation='elu', input_shape=(784,)))
autoencoder.add(Dense(128,  activation='elu'))
autoencoder.add(Dense(10,   activation='linear', name="bottleneck"))
autoencoder.add(Dense(128,  activation='elu'))
autoencoder.add(Dense(512,  activation='elu'))
autoencoder.add(Dense(784,  activation='sigmoid'))
autoencoder.compile(loss='mean_squared_error', optimizer = Adam())
trained_model = autoencoder.fit(train_x, train_x, batch_size=1024, epochs=10, verbose=1, validation_data=(val_x, val_x))
encoder = Model(autoencoder.input, autoencoder.get_layer('bottleneck').output)
encoded_data = encoder.predict(train_x)  # bottleneck representation
decoded_output = autoencoder.predict(train_x)        # reconstruction
encoding_dim = 10
# return the decoder
encoded_input = Input(shape=(encoding_dim,))
decoder = autoencoder.layers[-3](encoded_input)
decoder = autoencoder.layers[-2](decoder)
decoder = autoencoder.layers[-1](decoder)
decoder = Model(encoded_input, decoder)

### Anomaly Detection

In [ ]:
img = image.load_img("3. Training & Advanced Things/3.ipynb", target_size=(28, 28), color_mode = "grayscale")
input_img = image.img_to_array(img)
inputs = input_img.reshape(1,784)
target_data = autoencoder.predict(inputs)
dist = np.linalg.norm(inputs - target_data, axis=-1)
print(dist)